## Read Taxi Trip Data from ADLS Gen2
This notebook reads `taxi_tripdata.csv` from Azure Data Lake Storage Gen2 using the **ADLSManager** class defined in `ADLS_Databricks_Connection.py`.

**Pipeline steps:**
1. Load the ADLS connectivity module
2. Initialize `ADLSManager` with `etl_config.json`
3. Configure OAuth authentication
4. Read `taxi_tripdata.csv` from ADLS
5. Explore and display the data

In [0]:
# Load ADLSManager class from the .py module
module_path = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks/ADLS_Databricks_Connection.py"

with open(module_path, "r") as f:
    exec(f.read())

In [0]:
# Initialize ADLSManager with the shared ETL config
config_path = "/Workspace/Users/pavanreddy_adf@outlook.com/Azure_Databricks/etl_config.json"

adls = ADLSManager(config_path)
adls.configure_oauth()
adls.print_summary()

In [0]:
# Read the CSV file using the path defined in etl_config.json
csv_path = adls.file_cfg["csv"]  # ainput1/taxi_tripdata.csv

df_taxi = adls.read_file(csv_path, file_format="csv", header="true", inferSchema="true")
print(f"Rows: {df_taxi.count():,}  |  Columns: {len(df_taxi.columns)}")

In [0]:
# Print schema and show a sample
df_taxi.printSchema()
display(df_taxi)

In [0]:
# Quick summary statistics for numeric columns
display(df_taxi.summary())

## Data Quality Validations
Production-grade checks covering **nulls, domain integrity, range bounds, temporal consistency, and anomaly detection**.

Each rule returns a pass/fail with affected row counts.

In [0]:
from pyspark.sql import functions as F

# ── Null / missing analysis ──────────────────────────────────────────
total_rows = df_taxi.count()

null_counts = df_taxi.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_taxi.columns]
)

null_df = (
    null_counts
    .unpivot([], df_taxi.columns, "column_name", "null_count")
    .withColumn("total_rows", F.lit(total_rows))
    .withColumn("null_pct", F.round(F.col("null_count") / F.col("total_rows") * 100, 2))
    .orderBy(F.col("null_count").desc())
)

print(f"Total rows: {total_rows:,}")
display(null_df)

In [0]:
from pyspark.sql import functions as F
from functools import reduce

# ── Validation rule definitions ──────────────────────────────────────
# Each rule: (name, category, condition_expr that flags BAD rows)
rules = [
    # Null checks on critical columns
    ("NULL_vendor_id",        "completeness", F.col("VendorID").isNull()),
    ("NULL_pickup_datetime",  "completeness", F.col("lpep_pickup_datetime").isNull()),
    ("NULL_dropoff_datetime", "completeness", F.col("lpep_dropoff_datetime").isNull()),
    ("NULL_passenger_count",  "completeness", F.col("passenger_count").isNull()),
    ("NULL_fare_amount",      "completeness", F.col("fare_amount").isNull()),

    # Domain integrity
    ("INVALID_vendor_id",     "domain", ~F.col("VendorID").isin(1, 2) & F.col("VendorID").isNotNull()),
    ("INVALID_ratecode_id",   "domain", ~F.col("RatecodeID").isin(1, 2, 3, 4, 5, 6, 99) & F.col("RatecodeID").isNotNull()),
    ("INVALID_payment_type",  "domain", ~F.col("payment_type").isin(1, 2, 3, 4, 5, 6) & F.col("payment_type").isNotNull()),
    ("INVALID_trip_type",     "domain", ~F.col("trip_type").isin(1, 2) & F.col("trip_type").isNotNull()),
    ("INVALID_store_fwd",     "domain", ~F.col("store_and_fwd_flag").isin("Y", "N") & F.col("store_and_fwd_flag").isNotNull()),

    # Range / business bounds
    ("NEGATIVE_fare",         "range", F.col("fare_amount") < 0),
    ("NEGATIVE_tip",          "range", F.col("tip_amount") < 0),
    ("NEGATIVE_total",        "range", F.col("total_amount") < 0),
    ("ZERO_trip_distance",    "range", F.col("trip_distance") == 0),
    ("EXTREME_distance",      "range", F.col("trip_distance") > 200),
    ("EXTREME_fare",          "range", F.col("fare_amount") > 500),
    ("EXTREME_passengers",    "range", F.col("passenger_count") > 6),

    # Temporal consistency
    ("DROPOFF_before_pickup", "temporal", F.col("lpep_dropoff_datetime") < F.col("lpep_pickup_datetime")),
    ("ZERO_duration_trip",    "temporal", F.col("lpep_dropoff_datetime") == F.col("lpep_pickup_datetime")),
]

# ── Execute all rules ────────────────────────────────────────────────
total_rows = df_taxi.count()
results = []

for rule_name, category, condition in rules:
    fail_count = df_taxi.filter(condition).count()
    results.append((rule_name, category, fail_count, round(fail_count / total_rows * 100, 2),
                    "PASS" if fail_count == 0 else "FAIL"))

df_dq = spark.createDataFrame(results, ["rule_name", "category", "failed_rows", "fail_pct", "status"])
display(df_dq.orderBy(F.col("failed_rows").desc()))

In [0]:
from pyspark.sql import functions as F
from functools import reduce

# ── Aggregate DQ scores ──────────────────────────────────────────────
total_rules  = df_dq.count()
passed_rules = df_dq.filter(F.col("status") == "PASS").count()
failed_rules = total_rules - passed_rules
dq_score     = round(passed_rules / total_rules * 100, 1)

print("=" * 50)
print("        DATA QUALITY REPORT")
print("=" * 50)
print(f"Total rows       : {total_rows:,}")
print(f"Rules evaluated  : {total_rules}")
print(f"Rules passed     : {passed_rules}")
print(f"Rules failed     : {failed_rules}")
print(f"DQ Score         : {dq_score}%")
print("=" * 50)

# ── Category-level breakdown ─────────────────────────────────────────
print("\n--- Failures by Category ---")
display(
    df_dq.filter(F.col("status") == "FAIL")
    .groupBy("category")
    .agg(
        F.count("*").alias("failed_rules"),
        F.sum("failed_rows").alias("total_failed_rows")
    )
    .orderBy(F.col("total_failed_rows").desc())
)

# ── Quarantine bad records ───────────────────────────────────────────
critical_conditions = [
    F.col("fare_amount") < 0,
    F.col("total_amount") < 0,
    F.col("trip_distance") > 200,
    F.col("lpep_dropoff_datetime") < F.col("lpep_pickup_datetime"),
]

quarantine_filter = reduce(lambda a, b: a | b, critical_conditions)
df_quarantine = df_taxi.filter(quarantine_filter)
df_clean      = df_taxi.filter(~quarantine_filter)

print(f"\nQuarantined rows : {df_quarantine.count():,}")
print(f"Clean rows       : {df_clean.count():,}")
print("\n\u2705 df_clean and df_quarantine are ready for downstream use.")

## Medallion Architecture (Bronze → Silver → Gold)

| Layer | Purpose | Table | Details |
|-------|---------|-------|---------|
| **Bronze** | Raw ingestion | `taxi_bronze` | All 83,691 rows + audit columns, zero transformations |
| **Bronze** | Quarantine | `taxi_quarantine` | Bad records isolated by DQ rules |
| **Silver** | Cleansed & enriched | `taxi_silver` | Clean records + derived columns (duration, speed, flags) |
| **Gold** | Daily summary | `gold_daily_summary` | Revenue, trip counts, avg metrics by date |
| **Gold** | Hourly demand | `gold_hourly_demand` | Pickup patterns by hour of day |
| **Gold** | Top locations | `gold_top_locations` | Busiest pickup & dropoff zones |

All layers written as **Delta** to ADLS Gen2.

In [0]:
# ======================================================================
# Medallion Architecture — Path Configuration
# ======================================================================

# Base path from ADLSManager
base_path = adls.get_path("medallion")

paths = {
    # Bronze
    "bronze":          f"{base_path}/bronze/taxi_tripdata",
    "quarantine":      f"{base_path}/bronze/taxi_quarantine",
    # Silver
    "silver":          f"{base_path}/silver/taxi_tripdata",
    # Gold
    "daily_summary":   f"{base_path}/gold/daily_trip_summary",
    "hourly_demand":   f"{base_path}/gold/hourly_demand_pattern",
    "top_locations":   f"{base_path}/gold/top_locations",
}

for layer, path in paths.items():
    print(f"{layer:20s} → {path}")

In [0]:
from pyspark.sql import functions as F

# ======================================================================
# BRONZE LAYER — Raw data, zero business logic
# ======================================================================
# Principles:
#   - Preserve every row and column exactly as ingested
#   - Add audit metadata (_ingested_at, _source_file, _batch_id)
#   - Write as Delta for ACID guarantees and time travel

df_bronze = (
    df_taxi
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(adls.file_cfg["csv"]))
    .withColumn("_batch_id", F.lit(1))
)

# Write Bronze
df_bronze.write.format("delta").mode("overwrite").save(paths["bronze"])
print(f"✔ Bronze written: {df_bronze.count():,} rows → {paths['bronze']}")

# Write Quarantine (bad records from DQ checks)
df_quarantine_bronze = (
    df_quarantine
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_quarantine_reason", F.lit("DQ_CRITICAL_FAILURE"))
)

df_quarantine_bronze.write.format("delta").mode("overwrite").save(paths["quarantine"])
print(f"✔ Quarantine written: {df_quarantine_bronze.count():,} rows → {paths['quarantine']}")

In [0]:
from pyspark.sql import functions as F

# ======================================================================
# SILVER LAYER — Cleansed, standardized, enriched
# ======================================================================
# Transformations:
#   1. Start from df_clean (quarantine excluded)
#   2. Drop ehail_fee (100% null — zero information)
#   3. Derive: trip_duration_minutes, speed_mph, is_long_trip
#   4. Standardize: fill null passenger_count with 1
#   5. Add processing audit column

df_silver = (
    df_clean
    # Drop zero-value column
    .drop("ehail_fee")

    # Derived columns
    .withColumn(
        "trip_duration_minutes",
        F.round(
            (F.unix_timestamp("lpep_dropoff_datetime") - F.unix_timestamp("lpep_pickup_datetime")) / 60, 2
        )
    )
    .withColumn(
        "speed_mph",
        F.round(
            F.when(
                (F.col("trip_duration_minutes") > 0) & (F.col("trip_distance") > 0),
                F.col("trip_distance") / (F.col("trip_duration_minutes") / 60)
            ).otherwise(F.lit(None)),
            2
        )
    )
    .withColumn(
        "is_long_trip",
        F.when(F.col("trip_distance") > 10, F.lit(True)).otherwise(F.lit(False))
    )
    .withColumn(
        "pickup_date",
        F.to_date("lpep_pickup_datetime")
    )
    .withColumn(
        "pickup_hour",
        F.hour("lpep_pickup_datetime")
    )

    # Standardize nulls
    .fillna({"passenger_count": 1, "congestion_surcharge": 0.0, "RatecodeID": 1})

    # Audit
    .withColumn("_processed_at", F.current_timestamp())
)

# Write Silver
df_silver.write.format("delta").mode("overwrite").save(paths["silver"])
print(f"✔ Silver written: {df_silver.count():,} rows → {paths['silver']}")
print(f"  Columns: {len(df_silver.columns)} (added 5 derived, dropped ehail_fee)")
df_silver.printSchema()

In [0]:
from pyspark.sql import functions as F

# ======================================================================
# GOLD LAYER — Business-Level Aggregates
# ======================================================================

# ---------- Gold 1: Daily Trip Summary ----------
df_gold_daily = (
    df_silver
    .groupBy("pickup_date")
    .agg(
        F.count("*").alias("total_trips"),
        F.sum("total_amount").alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.sum(F.when(F.col("is_long_trip"), 1).otherwise(0)).alias("long_trips"),
        F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph"),
        F.round(F.avg("passenger_count"), 2).alias("avg_passengers"),
    )
    .orderBy("pickup_date")
)

df_gold_daily.write.format("delta").mode("overwrite").save(paths["daily_summary"])
print(f"✔ Gold daily summary: {df_gold_daily.count()} days → {paths['daily_summary']}")
display(df_gold_daily)

In [0]:
from pyspark.sql import functions as F

# ---------- Gold 2: Hourly Demand Pattern ----------
df_gold_hourly = (
    df_silver
    .groupBy("pickup_hour")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
    )
    .orderBy("pickup_hour")
)

df_gold_hourly.write.format("delta").mode("overwrite").save(paths["hourly_demand"])
print(f"✔ Gold hourly demand: {df_gold_hourly.count()} hours → {paths['hourly_demand']}")
display(df_gold_hourly)

In [0]:
from pyspark.sql import functions as F

# ---------- Gold 3: Top Locations ----------
df_top_pickup = (
    df_silver
    .groupBy("PULocationID")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    )
    .withColumn("location_type", F.lit("pickup"))
    .withColumnRenamed("PULocationID", "location_id")
)

df_top_dropoff = (
    df_silver
    .groupBy("DOLocationID")
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    )
    .withColumn("location_type", F.lit("dropoff"))
    .withColumnRenamed("DOLocationID", "location_id")
)

df_gold_locations = (
    df_top_pickup
    .unionByName(df_top_dropoff)
    .orderBy(F.col("trip_count").desc())
)

df_gold_locations.write.format("delta").mode("overwrite").save(paths["top_locations"])
print(f"✔ Gold locations: {df_gold_locations.count()} rows → {paths['top_locations']}")
display(df_gold_locations.limit(20))

In [0]:
# ======================================================================
# Pipeline Summary — Medallion Architecture
# ======================================================================

layers = [
    ("Source (CSV)",       paths["bronze"].replace("/bronze/taxi_tripdata", ""), df_taxi.count()),
    ("Bronze (raw)",       paths["bronze"],       spark.read.format("delta").load(paths["bronze"]).count()),
    ("Bronze (quarantine)",paths["quarantine"],    spark.read.format("delta").load(paths["quarantine"]).count()),
    ("Silver (cleansed)",  paths["silver"],        spark.read.format("delta").load(paths["silver"]).count()),
    ("Gold (daily)",       paths["daily_summary"], spark.read.format("delta").load(paths["daily_summary"]).count()),
    ("Gold (hourly)",      paths["hourly_demand"], spark.read.format("delta").load(paths["hourly_demand"]).count()),
    ("Gold (locations)",   paths["top_locations"], spark.read.format("delta").load(paths["top_locations"]).count()),
]

print("=" * 65)
print("         MEDALLION PIPELINE SUMMARY")
print("=" * 65)
for name, path, count in layers:
    print(f"{name:25s} | {count:>10,} rows")
print("=" * 65)
print("\n\u2705 Medallion architecture pipeline completed successfully!")